# Multi-task LSTM: emotions and topic

A shared BiLSTM encoder with two heads: 6 sigmoid outputs for emotions
(multi-label) and 12 softmax outputs for topic (single-label multiclass).

Evaluation protocol (shared with the other two notebooks via `evaluation.py`):

* 70 / 15 / 15 multi-label-stratified split
* the `Tokenizer` is fitted on **training text only**
* `Embedding(mask_zero=True)` marks padded timesteps (`pad_sequences` pads with
  `0`), Keras propagates that mask through the BiLSTM into `Attention`, and
  `GlobalAveragePooling1D` excludes padded steps from the pooled vector. A
  separate `Masking` layer would have been cancelled: `Embedding` without
  `mask_zero` returns `None` from `compute_mask` and discards the mask.
* early stopping monitors validation loss; thresholds are tuned on validation
* the test split is evaluated exactly once

This notebook records no training output in the committed file. Run it with
`PYTHONHASHSEED=42` (see README) and read `results/lstm_*.json` for the metrics.


In [ ]:
import os

import numpy as np
import pandas as pd
import tensorflow as tf
from bnlp import CleanText
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    Attention,
    Bidirectional,
    Dense,
    Embedding,
    GlobalAveragePooling1D,
    Input,
    LSTM,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

from evaluation import (
    EMOTIONS,
    SEED,
    best_thresholds,
    emotions_metrics,
    majority_baseline,
    save_metrics,
    set_global_seed,
    split_indices,
    topic_metrics,
)

os.makedirs("results", exist_ok=True)
set_global_seed(SEED)
tf.keras.backend.clear_session()


In [ ]:
df = pd.read_csv("Bangla Emotion Dataset.csv")

clean_text = CleanText(
    fix_unicode=True,
    unicode_norm=True,
    unicode_norm_form="NFKC",
    remove_url=True,
    remove_email=True,
    remove_emoji=True,
    remove_number=True,
    remove_digits=True,
    remove_punct=True,
    replace_with_url="",
    replace_with_email="",
    replace_with_number="",
    replace_with_digit="",
    replace_with_punct="",
)

df["clean_description"] = df["Data"].apply(clean_text)

y_emotions = df[EMOTIONS].to_numpy(dtype=int)
topic_encoder = LabelEncoder()
y_topic = topic_encoder.fit_transform(df["Topic"])

splits = split_indices(y_emotions, seed=SEED)
train_idx, val_idx, test_idx = splits["train"], splits["val"], splits["test"]
print({name: len(idx) for name, idx in splits.items()})


## Tokenizer fitted on training text only

`Tokenizer` learns its vocabulary from the training split. Validation and test
comments that contain words unseen in training fall back to `<OOV>`, which is
the behaviour production inference would see.


In [ ]:
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, lower=True, oov_token="<OOV>")
tokenizer.fit_on_texts(df["clean_description"].iloc[train_idx])


def encode(texts):
    sequences = tokenizer.texts_to_sequences(texts)
    return pad_sequences(sequences, maxlen=max_len)


X_train = encode(df["clean_description"].iloc[train_idx])
X_val = encode(df["clean_description"].iloc[val_idx])
X_test = encode(df["clean_description"].iloc[test_idx])

emotions_train, emotions_val, emotions_test = (
    y_emotions[train_idx],
    y_emotions[val_idx],
    y_emotions[test_idx],
)
topic_train, topic_val, topic_test = (
    y_topic[train_idx],
    y_topic[val_idx],
    y_topic[test_idx],
)

vocab_size = min(max_words, len(tokenizer.word_index) + 1)
print("vocab size:", vocab_size, "| train shape:", X_train.shape)
print("fraction of padded steps in train:", round(float((X_train == 0).mean()), 3))

baseline = majority_baseline(emotions_test)


## Model

`Embedding(..., mask_zero=True)` is what makes the mask exist: `pad_sequences`
pads with `0`, so index `0` marks padding. Keras then propagates that mask
through `Bidirectional(LSTM(return_sequences=True))` into `Attention`, and
`GlobalAveragePooling1D` averages only unmasked steps.

`GlobalMaxPooling1D` cannot be used here: in Keras 2.15 it has no mask support,
so padded steps would still win the max. `GlobalAveragePooling1D` implements
`call(inputs, mask=None)` and excludes padding.


In [ ]:
def build_model(vocab_size, max_len, num_emotions, num_topics):
    input_layer = Input(shape=(max_len,))
    embedding_layer = Embedding(vocab_size, 100, mask_zero=True)(input_layer)
    lstm_layer = Bidirectional(LSTM(64, return_sequences=True))(embedding_layer)
    attention_layer = Attention()([lstm_layer, lstm_layer])
    pooling_layer = GlobalAveragePooling1D()(attention_layer)

    emotions_output = Dense(num_emotions, activation="sigmoid", name="emotions")(pooling_layer)
    topics_output = Dense(num_topics, activation="softmax", name="topics")(pooling_layer)

    model = Model(inputs=input_layer, outputs=[emotions_output, topics_output])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss={"emotions": "binary_crossentropy", "topics": "sparse_categorical_crossentropy"},
        metrics={"emotions": ["accuracy"], "topics": ["accuracy"]},
    )
    return model


model = build_model(vocab_size, max_len, len(EMOTIONS), len(topic_encoder.classes_))
model.summary()


## Training

Early stopping restores the best-validation-loss weights, so the validation
split selects the epoch but is not reported as a test result.


In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True, verbose=1
)

history = model.fit(
    X_train,
    {"emotions": emotions_train, "topics": topic_train},
    validation_data=(X_val, {"emotions": emotions_val, "topics": topic_val}),
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=2,
)


## Final test evaluation

Metrics are computed explicitly from `model.predict` and named by their
definition, instead of being read out of `model.evaluate` by list position.


In [ ]:
val_emotion_scores, val_topic_scores = model.predict(X_val)
thresholds = best_thresholds(emotions_val, val_emotion_scores)
print("thresholds from validation:", dict(zip(EMOTIONS, np.round(thresholds, 2))))

test_emotion_scores, test_topic_scores = model.predict(X_test)

emotion_results = emotions_metrics(emotions_test, (test_emotion_scores >= thresholds).astype(int))
emotion_results["model"] = "bilstm+attention (tokenizer fitted on train)"
emotion_results["split"] = "test 15%, unseen during training and threshold selection"
emotion_results["thresholds"] = dict(zip(EMOTIONS, thresholds.tolist()))
emotion_results["majority_baseline"] = baseline

topic_results = topic_metrics(topic_test, test_topic_scores.argmax(axis=1), list(topic_encoder.classes_))
topic_results["model"] = "bilstm+attention (tokenizer fitted on train)"
topic_results["split"] = "test 15%, unseen during training"

print("emotions macro F1:", round(emotion_results["macro_f1"], 4))
print("emotions subset accuracy:", round(emotion_results["subset_accuracy"], 4))
print("emotions hamming loss:", round(emotion_results["hamming_loss"], 4))
print("model per-label accuracy:", round(emotion_results["per_label_accuracy"], 4))
print("majority per-label accuracy:", round(baseline["per_label_majority_accuracy"], 4))
print("topic accuracy:", round(topic_results["accuracy"], 4))
print("topic macro F1:", round(topic_results["macro_f1"], 4))


## Persist weights and metrics


In [ ]:
model.save_weights("results/lstm_weights.weights.h5")
tokenizer_json = tokenizer.to_json()
with open("results/lstm_tokenizer.json", "w", encoding="utf-8") as handle:
    handle.write(tokenizer_json)

save_metrics("results/lstm_emotions.json", emotion_results)
save_metrics("results/lstm_topic.json", topic_results)
save_metrics("results/lstm_baseline.json", baseline)
print("saved results/lstm_*.json, lstm_weights.weights.h5, lstm_tokenizer.json")


## Predictions on held-out text


In [ ]:
def predict_emotion_and_topic(text):
    cleaned = clean_text(text)
    padded = pad_sequences(tokenizer.texts_to_sequences([cleaned]), maxlen=max_len)
    emotion_scores, topic_scores = model.predict(padded, verbose=0)
    emotion_scores = emotion_scores[0]
    labels = [name for name, score, thr in zip(EMOTIONS, emotion_scores, thresholds) if score >= thr]
    topic = topic_encoder.inverse_transform([int(topic_scores[0].argmax())])[0]
    return {
        "Emotions": dict(zip(EMOTIONS, np.round(emotion_scores, 4))),
        "Predicted labels": labels,
        "Topic": topic,
    }


held_out_index = int(test_idx[0])
print("held-out row:", df.loc[held_out_index, "Data"][:80])
print("gold:", [name for name in EMOTIONS if df.loc[held_out_index, name] == 1], df.loc[held_out_index, "Topic"])
print("prediction:", predict_emotion_and_topic(df.loc[held_out_index, "Data"]))
